# 02 - Constructing the Winger Population

## Decoding Winger Playstyles

This notebook constructs the player population used for the winger playstyle analysis.

The source FBref dataset only categorises players into broad positional groups (`FW`, `MF`, `DF`, `GK`). This is insufficient for the project because the `FW` category contains both central forwards and wide attackers, while some wingers may also be categorised as midfielders.

Rather than identifying wingers from their statistical behaviour, which would introduce circularity into the later style analysis, an independent auxiliary historical FIFA/SoFIFA dataset is therefore used to identify wide players more precisely.

The objective of this notebook is therefore to:

1. load the audited FBref dataset,
2. identify the broad pool of potential attacking players,
3. enrich players with independent detailed position information,
4. define and document a winger inclusion rule,
5. apply playing-time requirements,
6. manually sanity-check the resulting population,
7. save the final winger dataset for later feature engineering and modelling.

No playstyle modelling is performed in this notebook.

## 1. Load the audited player dataset

The cleaned dataset produced in Notebook 01 is used as the starting point.

Each observation represents a **player × club × season** spell rather than necessarily a complete player-season.

In [241]:
import pandas as pd
import numpy as np

df = pd.read_csv("fbref_2017_2024_audited.csv")

print("Rows:", len(df))
print("Columns:", df.shape[1])

df

Rows: 18243
Columns: 63


,player,nation,pos,squad,comp,age,born,Matches Played,Minutes,Goals,...,Shots on Target %,Shots on Target per 90,Goals per shot,Goals per shot on target,Aerial Duels Won %,Shot Creating Actions per 90,Goal Creating Actions per 90,Crosses Stopped,season,player_id
0,Patrick van Aanholt,Netherlands,DF,Crystal Palace,Premier League,26,1990,28,2184,5,...,33.3,0.45,0.15,0.45,54.5,1.90,0.16,0.0,2017-2018,Patrick van Aanholt_1990
1,Rolando Aarons,England,MF,Newcastle Utd,Premier League,21,1995,4,139,0,...,0.0,0.00,0.00,0.00,25.0,0.65,0.00,0.0,2017-2018,Rolando Aarons_1995
2,Rolando Aarons,England,MF,Hellas Verona,Serie A,21,1995,11,517,0,...,0.0,0.00,0.00,0.00,37.5,0.87,0.00,0.0,2017-2018,Rolando Aarons_1995
3,Ignazio Abate,Italy,DF,Milan,Serie A,30,1986,17,1057,1,...,50.0,0.17,0.25,0.50,55.6,2.30,0.34,0.0,2017-2018,Ignazio Abate_1986
4,Aymen Abdennour,Tunisia,DF,Marseille,Ligue 1,27,1989,8,499,0,...,50.0,0.18,0.00,0.00,50.0,0.36,0.00,0.0,2017-2018,Aymen Abdennour_1989
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18238,Lovro Zvonarek,Croatia,FW,Bayern Munich,Bundesliga,18,2005,5,163,1,...,100.0,0.55,1.00,1.00,28.6,1.66,0.55,0.0,2023-2024,Lovro Zvonarek_2005
18239,Martin Ødegaard,Norway,MF,Arsenal,Premier League,24,1998,35,3091,8,...,28.0,0.61,0.08,0.29,25.0,6.41,0.67,0.0,2023-2024,Martin Ødegaard_1998
18240,Milan Đurić,Bosnia and Herzegovina,FW,Hellas Verona,Serie A,33,1990,20,1204,5,...,59.1,0.97,0.18,0.31,75.8,2.47,0.30,0.0,2023-2024,Milan Đurić_1990
18241,Milan Đurić,Bosnia and Herzegovina,FW,Monza,Serie A,33,1990,17,1257,4,...,23.1,0.43,0.15,0.67,68.3,0.93,0.07,0.0,2023-2024,Milan Đurić_1990


## 2. Build the broad attacking-player candidate pool

The source data only provides four broad positions.

Potential wingers may be listed as either:

- `FW` — forwards
- `MF` — midfielders

Both groups are retained at this stage.

No player is classified as a winger based on their statistical profile.

In [242]:
candidate_pool = df[df["pos"].isin(["FW", "MF"])].copy()

print("Candidate observations:", len(candidate_pool))

display(
    candidate_pool["pos"]
    .value_counts()
    .rename("observations")
    .to_frame()
)

Candidate observations: 10179


,observations
pos,
MF,5906
FW,4273


In [243]:
candidate_pool[["player_id", "player", "season", "squad", "comp", "pos", "Minutes"]].head(20)

,player_id,player,season,squad,comp,pos,Minutes
1,Rolando Aarons_1995,Rolando Aarons,2017-2018,Newcastle Utd,Premier League,MF,139
2,Rolando Aarons_1995,Rolando Aarons,2017-2018,Hellas Verona,Serie A,MF,517
6,Mehdi Abeid_1992,Mehdi Abeid,2017-2018,Dijon,Ligue 1,MF,1176
8,Tammy Abraham_1997,Tammy Abraham,2017-2018,Swansea City,Premier League,FW,1726
9,Amir Abrashi_1990,Amir Abrashi,2017-2018,Freiburg,Bundesliga,MF,850
11,Afriyie Acquah_1992,Afriyie Acquah,2017-2018,Torino,Serie A,MF,951
12,Charlie Adam_1985,Charlie Adam,2017-2018,Stoke City,Premier League,MF,411
18,Yacine Adli_2000,Yacine Adli,2017-2018,Paris S-G,Ligue 1,FW,8
21,Aritz Aduriz_1981,Aritz Aduriz,2017-2018,Athletic Club,La Liga,FW,2152
22,Ibrahim Afellay_1986,Ibrahim Afellay,2017-2018,Stoke City,Premier League,MF,166


## 3. Prepare the auxiliary positional data

The FBref source only identifies players as forwards, midfielders, defenders or goalkeepers and therefore cannot distinguish wingers from central attackers.

Detailed positional information is taken from the historical EA Sports / SoFIFA player dataset.

For this project, a **winger** is defined as a player whose primary listed position for the corresponding FIFA edition is one of:

- `LW` — Left Winger
- `RW` — Right Winger
- `LM` — Left Midfielder
- `RM` — Right Midfielder

`LM` and `RM` are included because older FIFA editions and traditional formations frequently represent wide attacking players as wide midfielders rather than forwards.

Only positional metadata is taken from the FIFA dataset. FIFA ability ratings and attributes are not used in the football analysis.

In [245]:
fifa = pd.read_csv("male_players.csv", low_memory=False)

fifa_positions = fifa[
    [
        "player_id",
        "short_name",
        "long_name",
        "fifa_version",
        "dob",
        "nationality_name",
        "club_name",
        "league_name",
        "player_positions",
        "club_position"
    ]
].copy()

fifa_positions["primary_position"] = (
    fifa_positions["player_positions"]
    .str.split(",")
    .str[0]
    .str.strip()
)

fifa_positions["birth_year"] = pd.to_datetime(
    fifa_positions["dob"]
).dt.year

In [246]:
fifa_positions = fifa_positions[
    fifa_positions["fifa_version"].between(18, 24)
].copy()

In [247]:
fifa_positions.head(5)

,player_id,short_name,long_name,fifa_version,dob,nationality_name,club_name,league_name,player_positions,club_position,primary_position,birth_year
0,231747,K. Mbappé,Kylian Mbappé Lottin,24,1998-12-20,France,Paris Saint Germain,Ligue 1,"ST, LW",LW,ST,1998
1,239085,E. Haaland,Erling Braut Haaland,24,2000-07-21,Norway,Manchester City,Premier League,ST,ST,ST,2000
2,192985,K. De Bruyne,Kevin De Bruyne,24,1991-06-28,Belgium,Manchester City,Premier League,"CM, CAM",SUB,CM,1991
3,158023,L. Messi,Lionel Andrés Messi Cuccittini,24,1987-06-24,Argentina,Inter Miami,Major League Soccer,"CF, CAM",RF,CF,1987
4,165153,K. Benzema,Karim Benzema,24,1987-12-19,France,Al Ittihad,Pro League,"CF, ST",RS,CF,1987


## 4. Align football seasons with FIFA editions

Each FBref domestic season is mapped to the FIFA edition released during that season.

For example, the 2017/18 season corresponds to FIFA 18 and the 2023/24 season corresponds to EA Sports FC 24.

In [248]:
season_to_fifa = {
    "2017-2018": 18,
    "2018-2019": 19,
    "2019-2020": 20,
    "2020-2021": 21,
    "2021-2022": 22,
    "2022-2023": 23,
    "2023-2024": 24
}

candidate_pool["fifa_version"] = (
    candidate_pool["season"]
    .map(season_to_fifa)
)

In [249]:
fifa_to_season = {
    version: season
    for season, version in season_to_fifa.items()
}

fifa_positions["season"] = (
    fifa_positions["fifa_version"]
    .map(fifa_to_season)
)

In [250]:
WINGER_POSITIONS = ["LW", "RW", "LM", "RM"]

fifa_positions["is_winger"] = (
    fifa_positions["primary_position"]
    .isin(WINGER_POSITIONS)
)

## 5. Match player identities between FBref and FIFA Datasets

The two datasets do not share a common player identifier and use different naming conventions.

For example, FBref may record `Mohamed Salah`, while FIFA records the same player as `Mohamed Salah Ghaly`.

Player matching is therefore constrained using:

1. corresponding season / FIFA edition,
2. birth year,
3. normalized name similarity,
4. club-name similarity as a secondary validation signal.

Restricting candidates by season and birth year substantially reduces the risk of incorrect fuzzy-name matches.

Match scores are retained so uncertain cases can be reviewed rather than silently accepted.

In [251]:
!pip install rapidfuzz -q

In [252]:
from rapidfuzz.fuzz import token_set_ratio, WRatio
import re
import unicodedata

In [253]:
def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char for char in text
        if not unicodedata.combining(char)
    )

    text = re.sub(r"[^a-z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [254]:
candidate_pool["name_norm"] = (
    candidate_pool["player"]
    .apply(normalize_text)
)

candidate_pool["club_norm"] = (
    candidate_pool["squad"]
    .apply(normalize_text)
)

fifa_positions["short_name_norm"] = (
    fifa_positions["short_name"]
    .apply(normalize_text)
)

fifa_positions["long_name_norm"] = (
    fifa_positions["long_name"]
    .apply(normalize_text)
)

fifa_positions["club_norm"] = (
    fifa_positions["club_name"]
    .apply(normalize_text)
)

In [256]:
candidate_pool[["name_norm","club_norm"]].head(5)

,name_norm,club_norm
1,rolando aarons,newcastle utd
2,rolando aarons,hellas verona
6,mehdi abeid,dijon
8,tammy abraham,swansea city
9,amir abrashi,freiburg


In [258]:
WINGER_POSITIONS = {"LW", "RW", "LM", "RM"}

fifa_positions["is_winger"] = (
    fifa_positions["primary_position"].isin(WINGER_POSITIONS)
    | fifa_positions["club_position"].isin(WINGER_POSITIONS)
)

In [263]:
fifa_positions[["long_name_norm","club_norm","primary_position","club_position","is_winger"]].sample(n=10)

,long_name_norm,club_norm,primary_position,club_position,is_winger
52047,jaime santos colado,palmaflor del tropico,RW,RW,True
96297,hugo martin nervo,santos laguna,CB,LCB,False
30888,joshua laws,wellington phoenix,CB,LCB,False
70662,alexander ruud tveter,sarpsborg 08,ST,SUB,False
32411,silas nwankwo,mjallby,ST,LS,False
125180,soren eismann,fc carl zeiss jena,CDM,SUB,False
129386,martin holm,horsens,RB,SUB,False
13853,sean patrick roughan,lincoln city,LWB,LCB,False
114557,jeronimo figueroa cabrera,las palmas,LM,SUB,True
112045,federico marchetti,lazio,GK,RES,False


In [264]:
def find_fifa_match(row, fifa_df):

    candidates = fifa_df[
        (fifa_df["fifa_version"] == row["fifa_version"])
        & (fifa_df["birth_year"] == row["born"])
    ].copy()

    if candidates.empty:
        return pd.Series({
            "fifa_player_id": pd.NA,
            "fifa_name": pd.NA,
            "fifa_club": pd.NA,
            "primary_position": pd.NA,
            "club_position": pd.NA,
            "is_winger": pd.NA,
            "name_score": np.nan,
            "club_score": np.nan,
            "match_score": np.nan
        })

    candidates["name_score"] = candidates.apply(
        lambda x: max(
            token_set_ratio(
                row["name_norm"],
                x["short_name_norm"]
            ),
            token_set_ratio(
                row["name_norm"],
                x["long_name_norm"]
            )
        ),
        axis=1
    )

    candidates["club_score"] = candidates[
        "club_norm"
    ].apply(
        lambda x: WRatio(row["club_norm"], x)
    )

    # Player identity is driven primarily by name.
    # Club acts only as a supporting/tie-breaking signal.
    candidates["match_score"] = (
        0.9 * candidates["name_score"]
        + 0.1 * candidates["club_score"]
    )

    best = candidates.sort_values(
        "match_score",
        ascending=False
    ).iloc[0]

    return pd.Series({
        "fifa_player_id": best["player_id"],
        "fifa_name": best["long_name"],
        "fifa_club": best["club_name"],
        "primary_position": best["primary_position"],
        "club_position": best["club_position"],
        "is_winger": best["is_winger"],
        "name_score": best["name_score"],
        "club_score": best["club_score"],
        "match_score": best["match_score"]
    })

In [265]:
fbref_identities = (
    candidate_pool
    .groupby(["player_id", "born"], as_index=False)
    .agg(
        player=("player", "first"),
        name_norm=("name_norm", "first"),
        fbref_clubs=(
            "club_norm",
            lambda x: list(pd.unique(x))
        )
    )
)

print("FBref observations:", len(candidate_pool))
print("Unique footballers to match:", len(fbref_identities))

FBref observations: 10179
Unique footballers to match: 3514


In [266]:
fifa_identities = (
    fifa_positions
    .groupby(["player_id", "birth_year"], as_index=False)
    .agg(
        fifa_name=("long_name", "last"),
        long_names=(
            "long_name_norm",
            lambda x: list(pd.unique(x))
        ),
        short_names=(
            "short_name_norm",
            lambda x: list(pd.unique(x))
        ),
        fifa_clubs=(
            "club_norm",
            lambda x: list(pd.unique(x))
        )
    )
)

In [267]:
from rapidfuzz import process, fuzz

In [268]:
name_choices_by_birth_year = {}

for birth_year, group in fifa_identities.groupby("birth_year"):

    choices = {}

    for row in group.itertuples():

        for i, name in enumerate(row.long_names):
            if name:
                choices[(row.player_id, "long", i)] = name

        for i, name in enumerate(row.short_names):
            if name:
                choices[(row.player_id, "short", i)] = name

    name_choices_by_birth_year[birth_year] = choices

In [269]:
fifa_clubs_lookup = (
    fifa_identities
    .set_index("player_id")["fifa_clubs"]
    .to_dict()
)

fifa_name_lookup = (
    fifa_identities
    .set_index("player_id")["fifa_name"]
    .to_dict()
)

### Creating the identity matcher

In [270]:
from rapidfuzz import process, fuzz

def match_fbref_identity(row):

    choices = name_choices_by_birth_year.get(row["born"])

    # No FIFA candidates with this birth year
    if not choices:
        return pd.Series({
            "fifa_player_id": pd.NA,
            "fifa_name": pd.NA,
            "name_score": np.nan,
            "club_score": np.nan,
            "name_tie_count": 0
        })

    # Find the strongest name candidates
    results = process.extract(
        row["name_norm"],
        choices,
        scorer=fuzz.token_set_ratio,
        limit=10
    )

    # A FIFA player may appear more than once because
    # we search both short_name and long_name.
    # Keep only that player's best name score.
    player_scores = {}

    for _, score, key in results:

        fifa_player_id = key[0]

        player_scores[fifa_player_id] = max(
            score,
            player_scores.get(fifa_player_id, 0)
        )

    # Find the highest name score achieved
    best_name_score = max(
        player_scores.values()
    )

    # Count how many DIFFERENT FIFA players
    # achieved that same best name score
    name_tie_count = sum(
        np.isclose(score, best_name_score)
        for score in player_scores.values()
    )

    ranked_candidates = []

    for fifa_player_id, name_score in player_scores.items():

        fifa_clubs = fifa_clubs_lookup.get(
            fifa_player_id, []
        )

        if row["fbref_clubs"] and fifa_clubs:

            club_score = max(
                fuzz.token_set_ratio(
                    fbref_club,
                    fifa_club
                )
                for fbref_club in row["fbref_clubs"]
                for fifa_club in fifa_clubs
            )

        else:
            club_score = 0

        ranked_candidates.append(
            (
                fifa_player_id,
                name_score,
                club_score
            )
        )

    # Highest name score wins.
    # Club similarity breaks ties.
    best = max(
        ranked_candidates,
        key=lambda x: (x[1], x[2])
    )

    fifa_player_id, name_score, club_score = best

    return pd.Series({
        "fifa_player_id": fifa_player_id,
        "fifa_name": fifa_name_lookup[fifa_player_id],
        "name_score": name_score,
        "club_score": club_score,
        "name_tie_count": name_tie_count
    })

### Run the identity matcher

Each unique FBref player is matched once to a stable FIFA player identity. Season-specific positional information will be attached only after identity matching is complete.

In [271]:
match_columns = [
    "fifa_player_id",
    "fifa_name",
    "name_score",
    "club_score",
    "name_tie_count"
]

fbref_identity_base = fbref_identities.drop(
    columns=match_columns,
    errors="ignore"
).copy()

identity_matches = fbref_identity_base.apply(
    match_fbref_identity,
    axis=1
)

fbref_identities = pd.concat(
    [
        fbref_identity_base.reset_index(drop=True),
        identity_matches.reset_index(drop=True)
    ],
    axis=1
)

In [272]:
fbref_identities[
    [
        "player",
        "fifa_name",
        "name_score",
        "club_score",
        "name_tie_count"
    ]
].head()

,player,fifa_name,name_score,club_score,name_tie_count
0,Aaron Connolly,Aaron Anthony Connolly,100.0,100.0,1
1,Aaron Hunt,Aaron Hunt,100.0,100.0,1
2,Aaron Lennon,Aaron Lennon,100.0,100.0,1
3,Aaron Leya Iseka,Aaron Leya Iseka,100.0,100.0,2
4,Aaron Mooy,Aaron Frank Mooy,100.0,100.0,1


## 6. Apply a minimum playing-time requirement

Player style should be inferred from a meaningful body of playing time rather than a small number of appearances.

A minimum threshold of **1,125 minutes** is applied to each player-club-season observation, equivalent to 12.5 full 90-minute matches.

Because the observational unit is player × club × season, the threshold is applied separately to each club spell.

In [273]:
MIN_MINUTES = 1125

eligible_pool = candidate_pool[
    candidate_pool["Minutes"] >= MIN_MINUTES
].copy()

print("Candidate observations:", len(candidate_pool))
print("Eligible observations:", len(eligible_pool))
print(
    "Eligible unique players:",
    eligible_pool["player_id"].nunique()
)

Candidate observations: 10179
Eligible observations: 5212
Eligible unique players: 1982


### Restrict identity validation to eligible players

Only players with at least 1,125 minutes in one or more player-club-season observations can enter the final analysis.

Identity-match validation is therefore restricted to this population.

In [274]:
eligible_player_ids = eligible_pool[
    "player_id"
].unique()

eligible_matches = fbref_identities[
    fbref_identities["player_id"].isin(
        eligible_player_ids
    )
].copy()

print(
    "Eligible player identities:",
    len(eligible_matches)
)

Eligible player identities: 1982


### Identity-match quality control

An identity is accepted automatically when:

- normalized token-based name similarity is 100, and
- only one FIFA player achieves that best score.

All other cases are flagged for review rather than being force-assigned.

Club history is retained as a supporting validation signal, but is not used as the primary identity criterion.

In [275]:
eligible_matches["match_status"] = np.where(
    (eligible_matches["name_score"] == 100)
    & (eligible_matches["name_tie_count"] == 1),
    "automatic",
    "review"
)

In [276]:
match_summary = (
    eligible_matches["match_status"]
    .value_counts()
    .rename_axis("Match Status")
    .reset_index(name="Players")
)

match_summary

,Match Status,Players
0,automatic,1813
1,review,169


In [277]:
review_matches = eligible_matches[
    eligible_matches["match_status"] == "review"
].copy()

print("Players requiring review:",len(review_matches))

Players requiring review: 169


In [281]:
review_export = review_matches[
    [
        "player_id",
        "player",
        "born",
        "fbref_clubs",
        "fifa_player_id",
        "fifa_name",
        "name_score",
        "club_score",
        "name_tie_count"
    ]
].copy()

review_export["decision"] = ""
review_export["resolved_fifa_player_id"] = ""
review_export["notes"] = ""

review_export.to_csv(
    "identity_matches_for_review.csv",
    index=False
)

Non-automatic identity matches were reviewed outside the notebook and stored in a small audit file.

Confirmed matches were retained, incorrect matches were corrected where possible, and unresolved FIFA identities were left unmatched.

In [282]:
manual_review = pd.read_csv(
    "identity_matches_for_review.csv"
)

manual_review["decision"].value_counts(dropna=False)

,count
decision,
accept,163
reject,6


In [283]:
assert manual_review["decision"].notna().all()

In [284]:
automatic_matches = eligible_matches[
    eligible_matches["match_status"] == "automatic"
][
    ["player_id", "fifa_player_id"]
].copy()

automatic_matches["match_method"] = "automatic"

In [285]:
manual_accepts = manual_review[
    manual_review["decision"] == "accept"
][
    ["player_id", "fifa_player_id"]
].copy()

manual_accepts["match_method"] = "manual"

In [286]:
identity_map = pd.concat(
    [
        automatic_matches,
        manual_accepts
    ],
    ignore_index=True
)

In [287]:
identity_map["fifa_player_id"] = pd.to_numeric(
    identity_map["fifa_player_id"],
    errors="coerce"
).astype("Int64")

In [288]:
print("Resolved identities:", len(identity_map))

assert not identity_map["player_id"].duplicated().any()

Resolved identities: 1976


In [289]:
eligible_pool = eligible_pool.merge(
    identity_map,
    on="player_id",
    how="left",
    validate="many_to_one"
)

In [290]:
print("Eligible observations:", len(eligible_pool))

print(
    "Without FIFA identity:",
    eligible_pool["fifa_player_id"].isna().sum()
)

Eligible observations: 5212
Without FIFA identity: 15


In [291]:
season_positions = fifa_positions[
    [
        "player_id",
        "fifa_version",
        "primary_position",
        "club_position",
        "is_winger"
    ]
].copy()

In [292]:
season_positions = season_positions.rename(
    columns={
        "player_id": "fifa_player_id"
    }
)

season_positions["fifa_player_id"] = (
    pd.to_numeric(
        season_positions["fifa_player_id"],
        errors="coerce"
    )
    .astype("Int64")
)

In [293]:
eligible_pool = eligible_pool.merge(
    season_positions,
    on=["fifa_player_id", "fifa_version"],
    how="left",
    validate="many_to_one"
)

## 7. Resolve missing positional classifications

FIFA is used as an auxiliary positional source rather than as an exhaustive record of all professional players.

A small number of eligible FBref observations could not receive a reliable season-specific positional classification from FIFA because the player was either absent from the corresponding FIFA edition or could not be matched confidently.

These remaining observations were manually verified using **historical season-specific Transfermarkt positional data**.

Classification was performed at the **player × club × season** level rather than assigning one permanent position to a player.

The same winger definition was retained:

- `LW`
- `RW`
- `LM`
- `RM`

Players whose historical role fell outside these positions were classified as non-wingers.

The manual decisions are stored separately in `manual_position_overrides.csv` to preserve an auditable record of the classifications.

In [294]:
lamine_mask = (
    (eligible_pool["player_id"] == "Lamine Yamal_2007")
    & (eligible_pool["season"] == "2023-2024")
)

eligible_pool.loc[
    lamine_mask,
    "primary_position"
] = "RW"

eligible_pool.loc[
    lamine_mask,
    "club_position"
] = "RW"

eligible_pool.loc[
    lamine_mask,
    "is_winger"
] = True

In [295]:
eligible_pool["position_source"] = np.where(
    eligible_pool["fifa_player_id"].notna(),
    "FIFA",
    pd.NA
)

eligible_pool.loc[
    lamine_mask,
    "position_source"
] = "Manual override"

In [296]:
unclassified = eligible_pool[
    eligible_pool["is_winger"].isna()
][
    [
        "player_id",
        "player",
        "season",
        "squad",
        "Minutes"
    ]
].drop_duplicates()

unclassified

,player_id,player,season,squad,Minutes
6,Lucas Alario_1992,Lucas Alario,2017-2018,Leverkusen,1550
309,Son Heung-min_1992,Son Heung-min,2017-2018,Tottenham,2317
444,Harrison Manzala_1994,Harrison Manzala,2017-2018,Amiens,1867
812,Hatem Ben Arfa_1987,Hatem Ben Arfa,2018-2019,Rennes,2033
860,Jonathan Calleri_1993,Jonathan Calleri,2018-2019,Alavés,2869
884,Samuel Chukwueze_1999,Samuel Chukwueze,2018-2019,Villarreal,1706
1053,Son Heung-min_1992,Son Heung-min,2018-2019,Tottenham,2039
1220,Virgil Misidjan_1993,Virgil Misidjan,2018-2019,Nürnberg,1543
1278,João Pedro_1992,João Pedro,2018-2019,Cagliari,2555
1336,Luca Rigoni_1984,Luca Rigoni,2018-2019,Parma,1761


In [297]:
unclassified_review = (
    eligible_pool[
        eligible_pool["is_winger"].isna()
    ][
        [
            "player_id",
            "player",
            "season",
            "squad",
            "Minutes"
        ]
    ]
    .drop_duplicates()
    .copy()
)

unclassified_review["manual_position"] = ""
unclassified_review["is_winger_manual"] = ""
unclassified_review["source"] = ""
unclassified_review["notes"] = ""

unclassified_review.to_csv(
    "manual_position_overrides.csv",
    index=False
)

In [298]:
manual_positions = pd.read_csv(
    "manual_position_overrides.csv"
)

manual_positions["is_winger_manual"] = (
    manual_positions["is_winger_manual"]
    .astype(str)
    .str.lower()
    .map({
        "true": True,
        "false": False
    })
)

In [299]:
eligible_pool = eligible_pool.merge(
    manual_positions[
        [
            "player_id",
            "season",
            "squad",
            "manual_position",
            "is_winger_manual",
            "source",
            "notes"
        ]
    ],
    on=["player_id", "season", "squad"],
    how="left",
    validate="one_to_one"
)

In [300]:
manual_mask = (
    eligible_pool["is_winger"].isna()
    & eligible_pool["is_winger_manual"].notna()
)

eligible_pool.loc[
    manual_mask,
    "is_winger"
] = eligible_pool.loc[
    manual_mask,
    "is_winger_manual"
]

eligible_pool.loc[
    manual_mask,
    "primary_position"
] = eligible_pool.loc[
    manual_mask,
    "manual_position"
]

In [301]:
eligible_pool["position_source"] = np.where(
    eligible_pool["fifa_player_id"].notna(),
    "FIFA",
    pd.NA
)

eligible_pool.loc[
    manual_mask,
    "position_source"
] = "Manual verification"

In [302]:
eligible_pool["is_winger"].isna().sum()

np.int64(0)

In [303]:
unclassified_count = (
    eligible_pool[
        "is_winger"
    ]
    .isna()
    .sum()
)

print(
    "Unclassified observations:",
    unclassified_count
)

Unclassified observations: 0


In [304]:
assert unclassified_count == 0

## 8. Construct the final winger population

After combining FIFA positional metadata with manually verified historical positions, every eligible player-club-season observation has a winger/non-winger classification.

The final analysis population retains observations classified as wide players and satisfying the minimum requirement of 1,125 minutes.

In [305]:
winger_pool = eligible_pool[
    eligible_pool[
        "is_winger"
    ].eq(True)
].copy()

In [306]:
print(
    "Winger observations:",
    len(winger_pool)
)

print(
    "Unique players:",
    winger_pool[
        "player_id"
    ].nunique()
)

print(
    "Seasons covered:",
    winger_pool[
        "season"
    ].nunique()
)

Winger observations: 1396
Unique players: 665
Seasons covered: 7


In [308]:
winger_pool[
    "season"
].value_counts().sort_index()

,count
season,
2017-2018,226
2018-2019,202
2019-2020,175
2020-2021,205
2021-2022,185
2022-2023,213
2023-2024,190


In [309]:
winger_pool[
    "primary_position"
].value_counts(
    dropna=False
)

,count
primary_position,
RM,396
LM,389
LW,215
RW,210
ST,67
CAM,59
CM,36
CF,19
LB,2


In [310]:
winger_pool[
    "position_source"
].value_counts()

,count
position_source,
FIFA,1380
Manual verification,15


Sanity Check and Preview sample

In [311]:
winger_pool[
    [
        "player",
        "season",
        "squad",
        "Minutes",
        "primary_position",
        "club_position",
        "position_source"
    ]
].sample(
    min(
        15,
        len(winger_pool)
    ),
    random_state=42
)

,player,season,squad,Minutes,primary_position,club_position,position_source
2928,Hwang Ui-jo,2020-2021,Bordeaux,2503,ST,LM,FIFA
3832,Álex Berenguer,2022-2023,Athletic Club,2297,RM,LM,FIFA
3263,Emmanuel Gyasi,2021-2022,Spezia,3035,RW,ST,FIFA
2300,Leonardo Bittencourt,2020-2021,Werder Bremen,1514,RM,SUB,FIFA
5096,Moses Simon,2023-2024,Nantes,1788,LM,LM,FIFA
2822,James Rodríguez,2020-2021,Everton,1764,CAM,RW,FIFA
4403,Crysencio Summerville,2022-2023,Leeds United,1426,RM,SUB,FIFA
2296,Steven Bergwijn,2020-2021,Tottenham,1208,LM,SUB,FIFA
1006,Moi Gómez,2018-2019,Huesca,3055,CAM,LM,FIFA
1559,Álex Berenguer,2019-2020,Torino,1823,LM,LF,FIFA


In [312]:
assert (
    winger_pool["Minutes"]
    >= MIN_MINUTES
).all()

assert (
    winger_pool[
        "is_winger"
    ]
    .eq(True)
    .all()
)

assert (
    winger_pool[
        "season"
    ]
    .isin(
        season_to_fifa.keys()
    )
    .all()
)

In [313]:
assert (
    winger_pool[
        [
            "player_id",
            "season",
            "squad",
            "comp"
        ]
    ]
    .duplicated()
    .sum()
    == 0
)

## 9. Save the analysis population

The resulting dataset contains the player-club-season observations used in the subsequent playstyle analysis.

This notebook only defines **who belongs in the winger population**.

No playstyle features, dimensionality reduction, similarity modelling or clustering are performed here. Those steps are handled separately in the following notebooks.

In [314]:
winger_pool.to_csv(
    "fbref_2017_2024_wingers.csv",
    index=False
)